# Algoritmo DPLL

El algoritmo **DPLL** es un algoritmo de **descenso recursivo** para determinar la **satisfactibilidad de una expresión booleana**.
A continuación te presentamos código en lenguaje Python para que puedas reforzar las definiciones necesarias para que puedas **implementar** el algoritmo.

## Literales

Una **literal** es una **variable proposicional afirmada o negada**.
La siguiente es una **clase de Python** que representa una **literal**.
El constructor recibe el nombre de una variable como primer argumento y si esta es afirmada (positive = True) o negada (positive = False).

Las literales negadas se representan como precedidas por una tilde (~).

In [2827]:
import re
class Literal:
    
    # regular expression for a valid variable name
    ID_REGEXP = re.compile(r"^[^\d\W]\w*\Z", re.UNICODE)
    
    """
    Sentencia proposicional que es una variable o una negación de la variable.
    --
    Propositional sentence tha is a variable or a negated variable
    """
    
    def __init__(self,variable,positive=True):
        """
        Crea o construye la literal de la variable dada
        :param variable: la variable de la literal
        :param positive: la literal positiva Si True, literal negada Si Falso
        --
        Creates the literal for the given variable
        :param variable: the variable of the literal
        :param positive: positive literal if True, negative literal if false

        """
        if re.match(Literal.ID_REGEXP,variable):
            self.variable = variable
        else:
            raise SyntaxError("Inválida nombre de la variable: '"+variable+"'")
        self.positive = positive
        
    @staticmethod
    def parse(s):
        s = s.replace(" ","")
        positive = not s.startswith("~")
        return Literal(s if positive else s[1:],positive)

    def __repr__(self):
        return self.__str__()
    
    def __str__(self):
        return ("" if self.positive else "~")+self.variable
        
    def __hash__(self):
        return hash(self.__str__())
    
    def __eq__(self,other):
        return self.__str__()==other.__str__()
    
    def __invert__(self):
        return Literal(self.variable,not self.positive)

Ejemplo para crear las literales $L_1=A$ y $L_2 = \neg B$

In [2828]:
l1 = Literal("A")
l2 = Literal("B",False)
l3 = Literal("C")
l4 = ~l3
print(l1,l2,l3,l4)

A ~B C ~C


## Cláusulas

Una **cláusula** es una **disyunción** (OR, |, ||) de **literales**.
La siguiente es una clase en Python para representar una **cláusula**.
El constructor recibe como primer argumento las literales que forman parte de la cláusula. Un segundo argumento del constructor indica cuando la representación de dichas literales es usando un conjunto que admite hashing.

In [2829]:
class Clause:
    """
    Una cláusula proposicional.
    Esta clase mantiene un patrón singleton de la cláusula con frozen hash
    para que cualquier simplificación se propague a los conjuntos y diccionarios
    en una sola operación.
    --
    A propositional clause
    This class keeps a singleton pattern of the clause with frozen hash
    so that any simplification can be propagated to sets and dictionaries
    in a single operation
    """
    
    def __init__(self, literals=None,frozen_hash=True):
        """
        Constructor de una cláusula con el conjunto de literales
        :param literals: el conjunto de literales
        :param frozen_hash: Si True el hash Si no basarse en el contenido
        --
        cretates the clause with the provided set of literals
        :param literals: the set of literals
        :param frozen_hash: if true the hash is not based on content
        """
        self.frozen_hash = frozen_hash
        if literals:
            self.literals = frozenset(literals) if frozen_hash else literals
        else:
            self.literals = set()
        
    def __hash__(self):
        """
        El número hash es la locación en la memoria cuando la bandera
        frozen_hash es True, de otra manera el hash es el mismo que el
        hash del frozenset interno con las literales.
        :returns: el número hash de la cláusula
        --
        the hash number is the memory location when the flag
        frozen_hash is True, otherwise the hash is the same as the 
        hash of the internal frozenset containing the literals
        :returns: the hash number of the clause
        """
        return id(self) if self.frozen_hash else hash(self.literals)
    
    def __eq__(self,other):
        """
        La equidad o igualdad entre diferentes instancias de la misma cláusula solo comprueba
        cuando frozen_hash es Falso. Si frozen_hash es True, el método
        regresará True, solo por la misma instancia de la clase.
        --
        Equality between different instances of the same clause is only checked
        when frozen_hash is False. If frozen_hash is True, the method will
        return True, only for the same instance of the class
        """
        if self.__hash__()==hash(other):
            if self.frozen_hash:
                return True
            else:
                return self.literals == other.literals
        else:
            return False
    
    def __iadd__(self,literal):
        """
        Agrega la literal a la cláusula
        :param literal: la literal para agregar
        --
        adds the literal to the clause
        :param literal: the literal to add
        """
        if isinstance(literal,Literal):
            self.literals = self.literals.union({literal})
            return self
        else:
            raise TypeError("An argument of type Literal was expected")
            
    def __add__(self,literal):
        """
        Agrega la literal a la cláusula
        :param literal: la literal para agregar
        --
        adds the literal to the clause
        :param literal: the literal to add
        """
        if isinstance(literal,Literal):
            return Clause(self.literals.union({literal}),self.frozen_hash)
        else:
            raise TypeError("An argument of type Literal was expected")
            
    def __isub__(self,literal):
        """
        Remueve la literal de la cláusula
        :param literal: la literal borrada
        --
        deletes a literal from the clause
        :param literal: the literal to substract
        """
        if isinstance(literal, Literal):
            self.literals = self.literals.difference({literal})
            return self
        else:
            raise TypeError("A literal of type string was expected")

    def __sub__(self,literal):
        """
        Remueve la literal de la cláusula
        :param literal: la literal borrada
        --
        deletes a literal from the clause
        :param literal: the literal to substract
        """
        if isinstance(literal, Literal):
            return Clause(self.literals.difference({literal}),self.frozen_hash)
        else:
            raise TypeError("A literal of type string was expected")
        
    def __str__(self):
        return "( "+" | ".join(map(str,self.literals))+" )"
    
    def __repr__(self):
        return self.__str__()
    
    def __iter__(self):
        return (i for i in self.literals)
    
    def __len__(self):
        return len(self.literals)
    
    def copy(self):
        return Clause(self.literals.copy(),self.frozen_hash)

Ejemplo creamos una cláusula $c_1$ formada por las literales $l_1$ y $l_2$

In [2830]:
c1 = Clause({l1,l2})
print(c1)

( A | ~B )


## Fórmula en Forma Normal Conjuntiva

In [2831]:
import logging as log
class FormulaCNF:
    """
    Fórmula en Forma Normal Conjuntiva
    :param formula: fórmula FNC  (string to parse)
    ejemplo: '(A|!B)&(!A|B|C)'
    --
    Formula in Conjunctive Normal Form
    :param formula: CNF formula (string to parse)
    example: '(A|!B)&(!A|B|C)'
    """
    def __init__(self,formula=None,assignment=None):
        #removes all white spaces
        if formula:
            formula = ''.join(formula.split())
            (self.variables,self.clauses) = self.parse(formula)
            self.build_dicts()  
            if assignment:
                self.assignment = assignment.copy()
            else:
                self.assignment = set()
            
            if log.getLogger().isEnabledFor(log.DEBUG):
                log.debug(self.string_internals())
                
    def __getitem__(self,literal):
        """
        notación corta para el método de simplificación
        :returns: la fórmula simplificada resultado de asurmir de que la literal
        es True
        --
        short notation for the simplify method
        :returns: the simplified formula resulting from assuming the literal
        is true
        """
        return self.simplify(literal)
            
    def empty_sentence(self):
        """
        :returns: True Si la fórmula no tiene cláusulas
        --
        :returns: true if the formula has no clauses
        """
        return not self.clauses
    
    def empty_clause(self):
        """
        :returns: True Si hay una cláusula vacía
        --
        :returns: true if there is an empty clause
        """
        return 0 in self.n_to_c
    
    def get_unit_clause_literal(self):
        """
        Obtiene la cláusula unitaria si no hay one, None entonces
        --
        Gets a unit clause if there is one, None otherwise
        """
        clause = next(iter(self.n_to_c[1])) \
        if 1 in self.n_to_c else {}
        return next(iter(clause)) if clause else None

    def get_pure_literal(self):
        """
        Obitiene la literal pura
        --
        Gest a pure literal
        """
        return next(iter(self.P)) if self.P else None
    
    def get_variable_literal(self):
        """
        Obtiene una variable de la formula
        --
        Get a variable literal
        """
        return Literal(next(iter(self.variables))) if self.variables else None

    def simplify(self,literal):
        """
        Simplifica la reciente sentencia asignando la literal
        :param literal: la litera a asumir como True
        --
        Simplifies the current sentence assming the provided literal
        :param literal: the literal to assume as true
        """
        log.debug("simplifying literal: "+str(literal))
        if not literal:
            raise ValueError(
                    "Invalid literal provided as argument: "+ str(literal))
        
        # stores the assignment
        self.assignment.add(literal)
        # deletes the variable of the literal
        self.variables = self.variables - {literal.variable}
        # updates the list of literals
        self.L = self.L -{literal}
        # updates the list of pure literals
        if self.isPureLiteral(literal):
            self.P = self.P - {literal}
        # deals with the clauses to delete
        if literal in self.l_to_c:
            for clause in self.l_to_c[literal].copy():
                self.remove_clause(clause)
                #TODO remover las cláusulas de otras entradas en l_to_c
                # update data structures for other literals
                for l in clause.copy():
                    self.decrease_literal_count(l)
                    self.del_from_dictionary_of_sets(
                            self.l_to_c,l,clause,True)
            #removes the entry from dicionary literal to clauses
            #del self.l_to_c[literal]
            
        # deals with the literals to delete negated literal
        neg_literal = ~literal 
        # updates list of literals
        self.L = self.L - {neg_literal}
        # deletes all negated literals from clauses
        if neg_literal in self.l_to_c:
            for clause in self.l_to_c[neg_literal]:
                self.remove_literal_from_clause(neg_literal,clause)
        # delete entry for negated literal from dictionary literal to clauses
        if neg_literal in self.l_to_c:
            del self.l_to_c[neg_literal]
        # updates list of negated literals
        if self.isPureLiteral(neg_literal):
            self.P = self.P - {neg_literal}

        log.debug("simplified formula:")
        if log.getLogger().isEnabledFor(log.DEBUG):
            log.debug(self.string_internals())
        return self
    
    def remove_clause(self,clause):
        """
        Remueve la cláusula de la estructura de datos
        :parama clause: la cláusula a borrar
        --
        Removes the clause from data structures
        :parama clause: the clause to delete
        """
        # deletes clause from inverse map counts
        self.clauses.remove(clause)
        n = len(clause)
        self.del_from_dictionary_of_sets(self.n_to_c,n,clause,True)
    
    def remove_literal_from_clause(self,literal,clause):
        # frozen_hash allows for unique clause 
        # all clauses are updated (they are the same clause) 
        m = len(clause)
        # remove clause from map of counts to clauses
        self.del_from_dictionary_of_sets(self.n_to_c,m,clause,True)
        # obtain simplified clause
        clause -= literal
        # update map of count to clauses
        self.add_to_dictionary_of_sets(self.n_to_c,m-1,clause)
        # update other counts related to the literal
        self.decrease_literal_count(literal)
        
                
    def decrease_literal_count(self,literal):
        # decrease number to literal counts
        # delete literal from previous count
        n = self.l_to_n[literal]
        self.del_from_dictionary_of_sets(self.n_to_l,n,literal,True)
        # update the literal count if it is not zero
        self.add_to_dictionary_of_sets(self.n_to_l,n-1,literal)
        
        # update counts for the literal
        if literal in self.l_to_n:
            self.l_to_n[literal] -= 1
        else:
            raise ValueError("Inconsistent counts for literal: "+str(literal))
            
    def isPureLiteral(self,literal):
        """
        :returns: True Si la literal es pura
        --
        :returns: true if the literal is pure
        """
        return literal in self.P
    
        
    def parse(self,formula):
        """
        Parses la fórmula FNC string
        """
        """
        Parses the CNF formula string
        """
        variables = set()
        clauses = set()
        clstr = re.findall("[^&]+",formula)
        for c in clstr:
            literals = re.findall("[^\(\)\|]+",c)
            clause = Clause()
            for l in literals:
                literal = Literal.parse(l)
                variables.add(literal.variable)
                clause += literal
            clauses.add(clause)
        return (variables,clauses)
    
    
    def build_dicts(self):
        """
        Contruye diccionarios para acceso eficiente
        a las cláusulas y literales
        l_to_c: diccionario, llaves son literales, valores son cláusulas
        n_to_c: diccionario, llaves son tamaños, valores son cláusulas
        l_to_n: diccionario, llaves son literales, valores son counts
        n_to_c: diccionario, llaves son counts, valores son cláusulas
        L: conjunto, todas las literales de la fórmula
        P: conjunto, todas las literales puras de la fórmula
        --
        Builds dictionaries for efficient access
        to clauses and literals
        l_to_c: dictionary, keys are literals, values are clauses
        n_to_c: dictionary, keys are sizes, values are clauses
        l_to_n: dictionary, keys are literals, values are counts
        n_to_c: dictionary, keys are counts, values are clauses
        L: set, all literals in the formula
        P: set, all pure literals in the formula
        """
        # W gives fast access to clauses by literals
        self.l_to_c = {}
        # n gives fast access to clauses by size
        self.n_to_c = {}
        # literal to counts
        self.l_to_n = {}
        # counts to literals
        self.n_to_l = {}
        
        for c in self.clauses:
            # build dictionary indexed by size
            self.add_to_dictionary_of_sets(self.n_to_c,len(c),c)
            for l in c:
                #builds dictionary indexed by literal
                self.add_to_dictionary_of_sets(self.l_to_c,l,c)
                #compute frecuency of literals
                self.l_to_n[l] = self.l_to_n[l] + 1 if l in self.l_to_n else 1
                
        #reverse literal counts
        self.L = set()
        for k,v in self.l_to_n.items():
            self.add_to_dictionary_of_sets(self.n_to_l,v,k)
            self.L.add(k)
        self.P = {v for v in self.L if ~v not in self.L}
    
    def add_to_dictionary_of_sets(self,d,k,v):
        """
        Agrega un valor al diccionario de conjuntos
        :param d: el diccionario
        :param k: la llaves
        :param v: el valor a agregar
        :param by_len: True Si la llaves es del tamaño del conjunto a agregar
        --
        Adds a value to a dictionary of sets
        :param d: the dictionary
        :param k: the key
        :param v: the value to add
        :param by_len: True if the key is the length of the set to add
        """
        singleton = {v}
        d[k] = d[k].union(singleton) if k in d else singleton
        
    def del_from_dictionary_of_sets(self,d,k,m,del_key):
        """
        Remueve el miembro conjunto del diccionario de conjuntos
        :param d: el diccionario
        :param k: la llaves
        :parma m: el miembro del conjunto a borrar
        :del_key: si esta bandera es True la entrada se remueve cuando el conjunto está vacío
        --
        Deletes a set member from a dictionary of sets
        :param d: the dictionary
        :param k: the key
        :parma m: the member of the set to delete
        :del_key: if this flag is True the entry is removed when set is empty
        """
        # removes with set difference
        d[k] = d[k] - {m}
        if del_key and not d[k]:
            del d[k]
    
    def copy(self):
        """
        Crea a copia de la fórmula FNC
        --
        Creates a shallow copy of the CNF formula
        """
        log.debug("Creating a copy of the current CNF formula")
        formula = FormulaCNF(str(self),self.assignment)
        return formula
    
    def string_internals(self):
        return ("clauses:"+str(self.clauses)+"\n"+
        "variables:"+str(self.variables)+"\n"+
        "assignment:"+str(self.assignment)+"\n"+
        "l_to_c:"+str(self.l_to_c)+"\n"+
        "n_to_c:"+str(self.n_to_c)+"\n"+
        "l_to_n:"+str(self.l_to_n)+"\n"+
        "n_to_l:"+str(self.n_to_l)+"\n"+
        "L:"+str(self.L)+"\n"+
        "P:"+str(self.P))
    
    def __str__(self):
        return " & ".join(map(str,self.clauses))
    
    def __repr__(self):
        return self.__str__()

<>:196: SyntaxWarning: invalid escape sequence '\('
<>:196: SyntaxWarning: invalid escape sequence '\('
/var/folders/dn/_5cs1ryn0gj52lsxggsjh_sh0000gn/T/ipykernel_37498/1214126863.py:196: SyntaxWarning: invalid escape sequence '\('
  literals = re.findall("[^\(\)\|]+",c)


Ejemplo de fórmula $\phi$ en FNC.

In [2832]:
phi = FormulaCNF("(A|~B)&(~A|C)&(~B|~C)&(C)")
print(phi)

( A | ~B ) & ( C ) & ( C | ~A ) & ( ~C | ~B )


### empty_sentence()

In [2833]:
#verificar si la formula es vacía
phi.empty_sentence()

False

### empty_clause()

In [2834]:
#¿tiene la formula una cláusula vacía?
phi.empty_clause()

False

### get_unit_clause_literal()

In [2835]:
#De existir una clúsula unitaria, obtiene la literal de la cláusula. Regresa None si no existe
l = phi.get_unit_clause_literal()
print(l)

C


### get_pure_literal()

In [2836]:
#Obtiene una literal pura de existir, None si no existe
l = phi.get_pure_literal()
print(l)

~B


### get_variable_literal()

In [2837]:
#Obtiene una literal con una variable arbitraria de la formula phi
v = phi.get_variable_literal()
print(v)

C


### copy()

In [2838]:
#Obtiene una copia de la formula
psi = phi.copy()
print(psi)

( A | ~B ) & ( C ) & ( C | ~A ) & ( ~C | ~B )


### simplify()

In [2839]:
phi.simplify(l)

( C ) & ( C | ~A )

In [2840]:
#Simplifica la formula asumiendo que la literal l es verdadera
psi = phi[l]
print(psi)

( C ) & ( C | ~A )


In [2841]:
#Obtiene las asignaciones aplicadas a una fórmula
a = phi.assignment
print(a)

{~B}


In [2842]:
class DPLL:
    """
    Algoritmo DPLL
    --
    DPLL algorithm
    """
    
    @staticmethod     
    def satisfiable(phi):
        """
        Determina si phi es satisfactible
        :param phi: una fórmula en FNC
        :returs: una tupla cuyo primer elemento indica
        --
        determines if phi is satisfiable
        :param phi: a CNF formula
        :returs: una tupla cuyo primer elemento indica
        si la fórmula es satisfactible o no, y el segundo
        la asignación que logró hacer la fórmula verdadera
        """
        log.info("phi: "+str(phi))             
        
        # Si la expresión de la fórmula phi o sentencia está vacía es satisfactoria.
        # Esdecir, todas las cláusulas se lograron. Regresa True
        if phi.empty_sentence():
            log.info("sentencia vacia")
            return (True, phi.assignment)
        #inserta tu código aquí     
        #if not phi.empty_sentence() & phi.empty_clause():
         #   log.info("cláusula vacia")
          #  return (False, None)

        else:
            # si phi no está vacía, no es satisfactoria
            #tuple = (False,None)
            tuple = (phi.empty_sentence(), phi.get_unit_clause_literal())
            return tuple 

        

In [2843]:
phi_1 = FormulaCNF(
    "(~x1 | x3 | x4) & (~x2 | x6 | x4) & (~x2 | ~x6 | ~x3) & (~x4 | ~x2) & (x2 | ~x3 | ~x1) & (x2 | x6 | x3) & (x2 | ~x6 | ~x4) & (x1 | x5) & (x1 | x6) & (~x6 | x3 | ~x5) & (x1 | ~x3 | ~x5)")
print(DPLL.satisfiable(phi_1))
print(phi_1)

(False, None)
( ~x2 | x6 | x4 ) & ( ~x2 | ~x4 ) & ( x6 | x2 | x3 ) & ( x1 | x5 ) & ( x6 | x1 ) & ( ~x5 | x1 | ~x3 ) & ( ~x1 | x4 | x3 ) & ( ~x2 | ~x6 | ~x3 ) & ( ~x1 | x2 | ~x3 ) & ( ~x6 | x2 | ~x4 ) & ( ~x5 | ~x6 | x3 )


In [2844]:
def pueba(phi):
    print('phi:',phi)
    print('empty_sentence, empty_clause:',phi.empty_sentence(), phi.empty_clause())
    unit_clause_literal = phi.get_unit_clause_literal()
    print('unit_clause_literal:',unit_clause_literal)
    pure_literal = phi.get_pure_literal()
    print('pure_literal:',pure_literal)
    variable_literal = phi.get_variable_literal()
    print('variable_literal:',variable_literal)
    phi_simpli = phi[variable_literal]
    print('phi_simplificada:',phi_simpli)
    assignment = phi_simpli.assignment
    print('assignment:',assignment)
    print('empty_sentence, empty_clause:',phi.empty_sentence(), phi.empty_clause())
pueba(psi)

phi: ( C ) & ( C | ~A )
empty_sentence, empty_clause: False False
unit_clause_literal: C
pure_literal: None
variable_literal: C
phi_simplificada: 
assignment: {C, ~B}
empty_sentence, empty_clause: True False


In [2845]:
DPLL.satisfiable(phi)

(True, {C, ~B})

In [2846]:
def simplificar_satisfactible(phi):
    print('¿Es satisfactible?', DPLL.satisfiable(phi))
    print('phi:',phi)
    print('empty_sentence, empty_clause:',phi.empty_sentence(), phi.empty_clause())
    unit_clause_literal = phi.get_unit_clause_literal()
    print('unit_clause_literal:',unit_clause_literal)
    pure_literal = phi.get_pure_literal()
    print('pure_literal:',pure_literal)
    variable_literal = phi.get_variable_literal()
    print('variable_literal:',variable_literal)
    print('')
    print('Simplificación con ϕ(',variable_literal,')')
    psi = phi[variable_literal]
    print('psi:',psi)
    unit_clause_literal = psi.get_unit_clause_literal()
    print('unit_clause_literal:',unit_clause_literal)
    pure_literal = psi.get_pure_literal()
    print('pure_literal:',pure_literal)
    variable_literal = psi.get_variable_literal()
    print(unit_clause_literal,pure_literal,variable_literal)
    print('')
    print('Simplificación con ϕ(',unit_clause_literal,')')
    psi2 = psi[unit_clause_literal]
    print('psi2:',psi2)    
    assignment = psi2.assignment
    print('assignment:',assignment)
    unit_clause_literal = psi2.get_unit_clause_literal()
    print('unit_clause_literal:',unit_clause_literal)
    pure_literal = psi2.get_pure_literal()
    print('pure_literal:',pure_literal)
    variable_literal = psi2.get_variable_literal()
    print(unit_clause_literal,pure_literal,variable_literal)
    print('')
    print('Simplificación con ϕ(',unit_clause_literal,')')
    psi3 = psi2[unit_clause_literal]
    print('psi3:',psi3)    
    assignment = psi3.assignment
    print('assignment:',assignment)
    unit_clause_literal = psi3.get_unit_clause_literal()
    print('unit_clause_literal:',unit_clause_literal)
    pure_literal = psi3.get_pure_literal()
    print('pure_literal:',pure_literal)
    variable_literal = psi3.get_variable_literal()
    print(unit_clause_literal,pure_literal,variable_literal)
    print('')
    print('Simplificación con ϕ(',unit_clause_literal,')')
    psi4 = psi3[unit_clause_literal]
    print('psi4:',psi4)    
    assignment = psi4.assignment
    print('assignment:',assignment)
    unit_clause_literal = psi4.get_unit_clause_literal()
    print('unit_clause_literal:',unit_clause_literal)
    pure_literal = psi4.get_pure_literal()
    print('pure_literal:',pure_literal)
    variable_literal = psi4.get_variable_literal()
    print(unit_clause_literal,pure_literal,variable_literal)
    print('')
    print('Simplificación con ϕ(',unit_clause_literal,')')
    psi4 = psi3[unit_clause_literal]
    print('psi4:',psi4)    
    assignment = psi3.assignment
    print('assignment:',assignment)
    unit_clause_literal = psi4.get_unit_clause_literal()
    print('unit_clause_literal:',unit_clause_literal)
    pure_literal = psi4.get_pure_literal()
    print('pure_literal:',pure_literal)
    variable_literal = psi4.get_variable_literal()
    print(unit_clause_literal,pure_literal,variable_literal)
    print('')
    print('Simplificación con ϕ(',unit_clause_literal,')')
    psi5 = psi4[unit_clause_literal]
    print('psi5:',psi5)    
    assignment = psi5.assignment
    print('assignment:',assignment)
    unit_clause_literal = psi5.get_unit_clause_literal()
    print('unit_clause_literal:',unit_clause_literal)
    pure_literal = psi5.get_pure_literal()
    print('pure_literal:',pure_literal)
    variable_literal = psi5.get_variable_literal()
    print(unit_clause_literal,pure_literal,variable_literal)
    print('empty_sentence, empty_clause:',phi.empty_sentence(), phi.empty_clause())
copia_phi_1 = phi_1.copy()
simplificar_satisfactible(phi_1)

¿Es satisfactible? (False, None)
phi: ( ~x2 | x6 | x4 ) & ( ~x2 | ~x4 ) & ( x6 | x2 | x3 ) & ( x1 | x5 ) & ( x6 | x1 ) & ( ~x5 | x1 | ~x3 ) & ( ~x1 | x4 | x3 ) & ( ~x2 | ~x6 | ~x3 ) & ( ~x1 | x2 | ~x3 ) & ( ~x6 | x2 | ~x4 ) & ( ~x5 | ~x6 | x3 )
empty_sentence, empty_clause: False False
unit_clause_literal: None
pure_literal: None
variable_literal: x2

Simplificación con ϕ( x2 )
psi: ( x6 | x4 ) & ( ~x4 ) & ( x1 | x5 ) & ( x6 | x1 ) & ( ~x5 | x1 | ~x3 ) & ( ~x1 | x4 | x3 ) & ( ~x6 | ~x3 ) & ( ~x5 | ~x6 | x3 )
unit_clause_literal: ~x4
pure_literal: None
~x4 None x5

Simplificación con ϕ( ~x4 )
psi2: ( x6 ) & ( x1 | x5 ) & ( x6 | x1 ) & ( ~x5 | x1 | ~x3 ) & ( ~x1 | x3 ) & ( ~x6 | ~x3 ) & ( ~x5 | ~x6 | x3 )
assignment: {x2, ~x4}
unit_clause_literal: x6
pure_literal: None
x6 None x6

Simplificación con ϕ( x6 )
psi3: ( x1 | x5 ) & ( ~x5 | x1 | ~x3 ) & ( ~x1 | x3 ) & ( ~x3 ) & ( x3 | ~x5 )
assignment: {x6, x2, ~x4}
unit_clause_literal: ~x3
pure_literal: None
~x3 None x3

Simplificación con ϕ(

In [2847]:
print(copia_phi_1)
print(copia_phi_1.empty_sentence(), copia_phi_1.empty_clause())
print(copia_phi_1.get_unit_clause_literal(), copia_phi_1.get_pure_literal(), copia_phi_1.get_variable_literal())
psi1_1 = copia_phi_1[copia_phi_1.get_variable_literal()]
print(psi1_1)
print(psi1_1.get_unit_clause_literal(), psi1_1.get_pure_literal(), psi1_1.get_variable_literal())
psi1_2 = psi1_1[psi1_1.get_unit_clause_literal()]
print(psi1_2)
print(psi1_2.get_unit_clause_literal(), psi1_2.get_pure_literal(), psi1_2.get_variable_literal())
psi1_3 = psi1_2[psi1_2.get_unit_clause_literal()]
print(psi1_3)
print(psi1_3.get_unit_clause_literal(), psi1_3.get_pure_literal(), psi1_3.get_variable_literal())
psi1_4 = psi1_3[psi1_3.get_unit_clause_literal()]
print(psi1_4)
print(psi1_4.get_unit_clause_literal(), psi1_4.get_pure_literal(), psi1_4.get_variable_literal())
print(psi1_4.empty_sentence(), psi1_4.empty_clause())
# Satisfactible
psi1_5 = psi1_4[psi1_4.get_unit_clause_literal()]
print(psi1_5)
print(psi1_5.get_unit_clause_literal(), psi1_5.get_pure_literal(), psi1_5.get_variable_literal())
psi1_6 = psi1_5[psi1_5.get_unit_clause_literal()]
print(psi1_6)
print(psi1_6.get_unit_clause_literal(), psi1_6.get_pure_literal(), psi1_6.get_variable_literal())
print(psi1_6.empty_sentence(), psi1_6.empty_clause())
print(psi1_5.assignment)

( ~x2 | ~x4 ) & ( x6 | x1 ) & ( ~x1 | x4 | x3 ) & ( ~x2 | ~x6 | ~x3 ) & ( ~x1 | x2 | ~x3 ) & ( ~x2 | x6 | x4 ) & ( x1 | x5 ) & ( x6 | x2 | x3 ) & ( x1 | ~x5 | ~x3 ) & ( ~x6 | x2 | ~x4 ) & ( ~x6 | x3 | ~x5 )
False False
None None x2
( ~x4 ) & ( x6 | x1 ) & ( ~x1 | x4 | x3 ) & ( ~x6 | ~x3 ) & ( x6 | x4 ) & ( x1 | x5 ) & ( x1 | ~x5 | ~x3 ) & ( ~x6 | x3 | ~x5 )
~x4 None x5
( x6 | x1 ) & ( ~x1 | x3 ) & ( ~x6 | ~x3 ) & ( x6 ) & ( x1 | x5 ) & ( x1 | ~x5 | ~x3 ) & ( ~x6 | x3 | ~x5 )
x6 None x6
( ~x1 | x3 ) & ( ~x3 ) & ( x1 | x5 ) & ( x1 | ~x5 | ~x3 ) & ( ~x5 | x3 )
~x3 None x3
( ~x1 ) & ( x1 | x5 ) & ( ~x5 )
~x1 None x1
False False
( x5 ) & ( ~x5 )
~x5 None x5
(  )
None None None
False True
{x2, ~x4, ~x5, ~x1, x6, ~x3}


In [2848]:
phi_2 = FormulaCNF("(A | B | C) & (~A | ~B)")
print('¿Es satisfactible?', DPLL.satisfiable(phi_2))
print('¿Está vacía la fórmula?',phi_2.empty_sentence())
print('fórmula ϕ =',phi_2)

print('Cláusula unitaria:',phi_2.get_unit_clause_literal())
print('Literal pura:',phi_2.get_pure_literal())
print('Simplificación con ϕ(',phi_2.get_pure_literal(),')')
psi3 = phi[phi_2.get_pure_literal()]
print('psi3:',phi[phi_2.get_pure_literal()])
#print(phi_3.get_variable_literal())
#print(psi3[phi_3.get_variable_literal()])
print('¿Está vacía la fórmula?',psi3.empty_sentence())
print('¿Está vacía la cláusula?',psi3.empty_clause())
print('¿Es satisfactible?', DPLL.satisfiable(psi3))

¿Es satisfactible? (False, None)
¿Está vacía la fórmula? False
fórmula ϕ = ( C | B | A ) & ( ~B | ~A )
Cláusula unitaria: None
Literal pura: C
Simplificación con ϕ( C )
psi3: 
¿Está vacía la fórmula? True
¿Está vacía la cláusula? False
¿Es satisfactible? (True, {C, ~B})


In [2851]:
#phi = FormulaCNF("(A|~B)&(~A|C)&(~B|~C)&(C)")
phi_3 = FormulaCNF("(A|~B)&(~A|C)&(~B|~C)&(C)")
print(phi_3)

( A | ~B ) & ( C | ~A ) & ( C ) & ( ~C | ~B )


## Ejercicios

In [2849]:
# Tuplas
tuple = (False,None)
print(type(tuple))
tuple

<class 'tuple'>


(False, None)

In [2850]:
mytuple = ("apple", "banana", "cherry")
print(type(mytuple))
mytuple

<class 'tuple'>


('apple', 'banana', 'cherry')